# v11 S (GPU 1): module sizes per layer + the paper-format co-activation figure (no training)
- **Sizes:** self and task size in every layer, for many checkpoints (phases <= 60), so they can be averaged over cycles. Self found two ways: on all the model's languages, and on Hindi only.
  distant: phases 10-12, 25-27, 40-42, 55-57 · Hindi-only and mixed: 10, 25, 30, 40, 45, 55, 60 · related: 5-16.
  Checkpoints downloaded only for this are deleted again afterwards to save disk.
- **Matrix:** for each model, each layer: the co-activation matrix reordered into groups (self block boxed), and below it every neuron's persistence to the next phase.
  distant 40 -> 41 · Hindi-only 40 -> 41 · mixed 40 -> 41 · related 13 -> 14.
Results: `analysis/v11/sizes/`, `analysis/v11/matrix/` · progress: `tail -f logs/v11_S_sizes_matrix.log`

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "1"; LOG = "v11_S_sizes_matrix"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, *cmd):
    """one call of core_v11.py; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", "core_v11.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(part, out=None):
    cmd = [sys.executable, "core_v11.py", "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-12000:]); print(r.stderr[-3000:])

DIST, HIN, MIX, REL = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0"

In [ ]:
if os.path.exists("analysis/v11/sizes/sizes.csv"):
    os.remove("analysis/v11/sizes/sizes.csv")                      # no resume: start clean
KEEP = {DIST: [10, 11, 12, 20, 30, 40, 41, 42], HIN: [10, 30, 40, 41, 45, 60], MIX: [10, 30, 40, 41, 45, 60], REL: list(range(0, 61))}
PLAN = [("distant", "distant", DIST, [10, 11, 12, 25, 26, 27, 40, 41, 42, 55, 56, 57]), ("distant", "hindi", HIN, [10, 25, 30, 40, 45, 55, 60]),
        ("distant", "mixed", MIX, [10, 25, 30, 40, 45, 55, 60]), ("indic", "related", REL, list(range(5, 17)))]
for langset, fam, r, phs in PLAN:
    for ph in phs:
        had = have(r, ph)
        if get(r, [ph]):
            run(langset, "--part", "sizes", "--family", fam, "--model", f"{fam}={r}:{ph}")
            if not had and ph not in KEEP[r]:
                for f in glob.glob(f"runs/{r}/ckpt/p{ph:03d}_*.pt"):
                    os.remove(f)
summary("sizes")

In [ ]:
for f in ["analysis/v11/matrix/persistence_per_neuron.csv"]:
    if os.path.exists(f): os.remove(f)
for langset, fam, r, ph in [("distant", "distant", DIST, 40), ("distant", "hindi", HIN, 40), ("distant", "mixed", MIX, 40), ("indic", "related", REL, 13)]:
    if len(get(r, [ph, ph + 1])) == 2:
        run(langset, "--part", "matrix", "--family", fam, "--model", f"{fam}={r}:{ph}")
for f in ["matrix_distant_dec1.png", "matrix_hindi_dec1.png", "matrix_related_dec1.png"]:
    if os.path.exists(f"analysis/v11/matrix/{f}"): display(Image(f"analysis/v11/matrix/{f}"))